In [ ]:
#!/usr/bin/env python3

# ── Library Check / Install ───────────────────────────────────────────────────
import importlib
import subprocess
import sys

def check_or_install(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        importlib.import_module(import_name)
        print(f"✓ {package_name}")
    except ImportError:
        print(f"✗ {package_name} not found — installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])
        print(f"✓ {package_name} installed")

required = [
    ("torch",        "torch"),
    ("torchvision",  "torchvision"),
    ("opacus",       "opacus"),
    ("numpy",        "numpy"),
    ("tqdm",         "tqdm"),
    ("matplotlib",   "matplotlib"),
]

print("Checking required libraries...\n")
for pip_name, import_name in required:
    check_or_install(pip_name, import_name)
print("\nAll libraries ready.\n")

# ── Imports ───────────────────────────────────────────────────────────────────
import os
import gc
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from opacus import GradSampleModule
from opacus.validators import ModuleValidator
from opacus.accountants import create_accountant
from tqdm import tqdm
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
# Set to single values for a quick single-run test.
# (Original sweep values are commented alongside for reference.)
BATCH_SIZE    = 512
NUM_EPOCHS    = 5
NOISE_MULT    = 0.6            # single value for testing (orig sweep: 0.5-2.0)
MAX_GRAD_NORM = 1.0            # C
DELTA         = 1e-5
RESULTS_PATH  = "cifar10_macadam_results.json"
seeds         = [42]           # single value for testing (orig sweep: [42,123,456,789,999])

# Learning rates
ETA           = 0.001          # Adam-based algorithms

# Adam hyperparameters
ADAM_BETA1    = 0.9
ADAM_BETA2    = 0.999
GAMMA_PRIME   = 1e-8           # DP-MACADAM-BC only (u_hat de-biasing floor)

# DP-MACADAM hyperparameters
H1            = 1e-12          # floor/ceiling refs used in diagnostics
H2            = 1e12
H1_           = 1e-4           # variance EMA floor (note: BC variant in notebook used 1e-10)
BETA3         = 0.999          # variance EMA decay
GAMMA1        = 1e-4           # stability constant for centering step (omitted)
GAMMA2        = 1e-8           # stability constant for Adam denominator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Seed ──────────────────────────────────────────────────────────────────────
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Dataset ───────────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print(f"\nTrain size:    {len(train_dataset):,}")
print(f"Test size:     {len(test_dataset):,}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Batches/epoch: {len(train_loader)}")
print(f"Seeds:         {seeds}")
print(f"Epochs:        {NUM_EPOCHS}")

# ── Privacy Accounting ────────────────────────────────────────────────────────
SAMPLE_RATE = BATCH_SIZE / len(train_dataset)
T_total     = NUM_EPOCHS * len(train_loader)
accountant  = create_accountant("prv")
accountant.history = [(NOISE_MULT, SAMPLE_RATE, T_total)]
eps = accountant.get_epsilon(delta=DELTA)
print(f"\nPrivacy: ε ≈ {eps:.2f}, δ = {DELTA} (Connect-the-Dots / PRV)\n")

# ── Model ─────────────────────────────────────────────────────────────────────
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

def make_fresh_model():
    m = ConvNet().to(device)
    m.train()
    return m

print(f"Model parameters: {sum(p.numel() for p in ConvNet().parameters()):,}")

# ── Helpers ───────────────────────────────────────────────────────────────────
def free_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

def save_results(results_to_add, filepath):
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            results = json.load(f)
    else:
        results = {}
    results.update(results_to_add)
    with open(filepath, "w") as f:
        json.dump(results, f)
    print(f"Saved. Keys in file: {list(results.keys())}")

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = model(x_batch)
            loss   = loss_fn(logits, y_batch)
            total_loss    += loss.item() * len(x_batch)
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_samples += len(x_batch)
    model.train()
    return total_loss / total_samples, total_correct / total_samples

def make_grad_sample_model(model):
    model.train()
    errors = ModuleValidator.validate(model, strict=False)
    if errors:
        print(f"Fixing model compatibility: {errors}")
        model = ModuleValidator.fix(model)
    return GradSampleModule(model)

def get_per_sample_grads(gs_model, x_batch, y_batch, loss_fn):
    gs_model.zero_grad()
    out  = gs_model(x_batch)
    loss = loss_fn(out, y_batch)
    loss.backward()
    G = torch.cat([
        p.grad_sample.flatten(start_dim=1)
        for p in gs_model.parameters()
        if p.grad_sample is not None
    ], dim=1)
    for p in gs_model.parameters():
        p.grad_sample = None
    return G

def apply_flat_update(model, update, eta):
    idx = 0
    with torch.no_grad():
        for p in model.parameters():
            numel = p.numel()
            p -= eta * update[idx:idx + numel].reshape(p.shape)
            idx += numel

loss_fn = nn.CrossEntropyLoss()


Checking required libraries...

✓ torch
✓ torchvision
✓ opacus
✓ numpy
✓ tqdm
✓ matplotlib

All libraries ready.

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
Files already downloaded and verified
Files already downloaded and verified

Train size:    50,000
Test size:     10,000
Batch size:    512
Batches/epoch: 98
Seeds:         [42]
Epochs:        5

Privacy: ε ≈ 5.88, δ = 1e-05 (Connect-the-Dots / PRV)

Model parameters: 582,346


# Find "optimal" $h1, h2$

In [ ]:
# ── Algorithm 3: DP-MACADAM, paper's v-fix + fully corrected kappa_t ─────────
# Same as Algorithm 2 (fixed v, single-beta EMA), but replaces the paper's
# closed-form kappa_t = 2(beta1-beta1^t)/(1+beta1) — which implicitly assumes
# m_hat shares one global coefficient family — with the exact, per-step
# normalized kappa_t, verified against brute-force simulation.
#
# kappa_t = (1 - beta1^t) * A_correct(t), where
#   A_correct(t) = 1 + sum_{k=1}^t c_k^(t) * ( S2^(k) - 2*c_k^(k) )
#   c_k^(t) = (1-beta1) * beta1^(t-k) / (1 - beta1^t)
#   c_k^(k) = (1-beta1) / (1 - beta1^k)
#   S2^(k)  = (1-beta1)*(1+beta1^k) / ((1+beta1)*(1-beta1^k))
#
# kappa_t depends only on (beta1, t) -- precompute once for all t up to T_total.
def precompute_kappa(T, beta1):
    kappa = np.zeros(T + 1)  # 1-indexed; kappa[0] unused
    for t in range(1, T + 1):
        k_idx = np.arange(1, t + 1)
        c_kt = (1 - beta1) * beta1 ** (t - k_idx) / (1 - beta1 ** t)
        c_kk = (1 - beta1) / (1 - beta1 ** k_idx)
        S2_k = (1 - beta1) * (1 + beta1 ** k_idx) / ((1 + beta1) * (1 - beta1 ** k_idx))
        A_t = 1 + np.sum(c_kt * (S2_k - 2 * c_kk))
        kappa[t] = (1 - beta1 ** t) * A_t
    return kappa

# Precompute for the full run length (epochs * batches/epoch)
NUM_EPOCHS = 5

T_total_run = NUM_EPOCHS * len(train_loader)
kappa_lookup = precompute_kappa(T_total_run, ADAM_BETA1)
kappa_lookup_t = torch.tensor(kappa_lookup, device=device, dtype=torch.float32)

def run_dpadam_adaclip_corrected_kappa(seed):
    set_seed(seed)
    model    = make_fresh_model()
    gs_model = make_grad_sample_model(model)
    d        = sum(p.numel() for p in model.parameters())
    m  = torch.zeros(d, device=device)
    u  = torch.zeros(d, device=device)
    s2 = torch.zeros(d, device=device)
    b  = torch.full((d,), MAX_GRAD_NORM / (d ** 0.25), device=device)
    m_hat_prev = torch.zeros(d, device=device)
    eval_accs = []
    t = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        for x_batch, y_batch in tqdm(train_loader, desc=f"  DP-MACAdam-corrected-kappa epoch {epoch}"):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            B = len(x_batch)
            t += 1
            noise_scale = NOISE_MULT * MAX_GRAD_NORM / B
            G = get_per_sample_grads(gs_model, x_batch, y_batch, loss_fn)
            W = (G - m_hat_prev) / b
            norms = torch.norm(W, dim=1, keepdim=True)
            W_bar = W / torch.clamp(norms, min=1.0)
            g_tilde = b * (W_bar.mean(dim=0) +
                      torch.randn(d, device=device) * noise_scale) \
                      + m_hat_prev
            m = ADAM_BETA1 * m + (1 - ADAM_BETA1) * g_tilde
            u = ADAM_BETA2 * u + (1 - ADAM_BETA2) * g_tilde ** 2
            m_hat  = m / (1 - ADAM_BETA1 ** t)
            u_hat  = u / (1 - ADAM_BETA2 ** t)
            update = m_hat / (torch.sqrt(u_hat) + GAMMA2)
            apply_flat_update(model, update, ETA)

            v      = (g_tilde - m_hat) ** 2
            s2     = ADAM_BETA1 * s2 + (1 - ADAM_BETA1) * v
            kappa_t = kappa_lookup_t[t].clamp(min=1e-12)    # kappa_0 = 0 causes nan
            # s2_hat = torch.clamp(
            #     s2 / kappa_t - b ** 2 * noise_scale ** 2,
            #     min=H1, max=H2)
            s2_hat = torch.clamp(
                s2 / kappa_t - b ** 2 * noise_scale ** 2,
                min=5e-5, max=1)
            s      = torch.sqrt(s2_hat)
            b      = torch.sqrt(s) * torch.sqrt(s.sum())
            m_hat_prev = m_hat.detach().clone()

        with torch.no_grad():
            print(f"\n  v      | min={v.min():.2e} max={v.max():.2e} "
                  f"mean={v.mean():.2e}")
            print(f"  kappa_t (t={t}) = {kappa_t.item():.6f}")
            print(f"  s2_hat | min={s2_hat.min():.2e} max={s2_hat.max():.2e} "
                  f"mean={s2_hat.mean():.2e} "
                  f"frac_at_floor={(s2_hat <= H1 * 1.01).float().mean():.3f}")
            print(f"  b      | min={b.min():.2e} max={b.max():.2e} mean={b.mean():.2e}")

        _, acc = evaluate(model, test_loader, loss_fn, device)
        eval_accs.append(acc)
        print(f"  Epoch {epoch} | Acc: {acc:.4f}")

    free_memory()
    return eval_accs


In [ ]:
# ── Run: single sigma, single seed, all 3 algorithm variants ─────────────────
# Set sigmas / seeds to lists with more values to reproduce a full sweep.
sigmas = [NOISE_MULT]   # e.g. [0.6]
seeds  = [42]             # single seed for a quick test run

NUM_EPOCHS = 5

algorithms = {
    # "dp-macadam-paper":          run_dpadam_adaclip_paper,           # paper's algorithm exactly
    "dp-macadam-corrected-kappa": run_dpadam_adaclip_corrected_kappa, # paper's v-fix + exact kappa_t
}

for sigma in sigmas:
    print(f"\nNoise scale: {sigma}")
    NOISE_MULT = sigma
    RESULTS_PATH = f"cifar10_macadam_variants_sigma{str(sigma).replace('.', '_')}.json"

    for algo_name, algo_fn in algorithms.items():
        print(f"\n{'='*60}")
        print(f"Algorithm: {algo_name}")
        print(f"{'='*60}")
        all_accs = []
        for seed in seeds:
            print(f"\n--- Seed {seed} ---")
            accs = algo_fn(seed)
            all_accs.append(accs)
        save_results({algo_name: all_accs}, RESULTS_PATH)
        free_memory()

    print("\n" + "="*60)
    print("All experiments complete.")
    print(f"Results saved to: {RESULTS_PATH}")
    print("="*60)



Noise scale: 0.6

Algorithm: dp-macadam-corrected-kappa

--- Seed 42 ---


  DP-MACAdam-corrected-kappa epoch 1: 100%|██████████| 98/98 [00:03<00:00, 26.34it/s]


  v      | min=2.17e-19 max=4.36e-03 mean=7.60e-05
  kappa_t (t=98) = 0.852565
  s2_hat | min=5.00e-05 max=6.00e-04 mean=5.00e-05 frac_at_floor=0.000
  b      | min=5.40e+00 max=1.00e+01 mean=5.40e+00


  Epoch 1 | Acc: 0.4535


  DP-MACAdam-corrected-kappa epoch 2: 100%|██████████| 98/98 [00:03<00:00, 26.06it/s]

...


  v      | min=1.82e-16 max=1.89e-03 mean=7.60e-05
  kappa_t (t=392) = 0.852632
  s2_hat | min=5.00e-05 max=2.13e-04 mean=5.00e-05 frac_at_floor=0.000
  b      | min=5.40e+00 max=7.75e+00 mean=5.40e+00


  Epoch 4 | Acc: 0.5640


  DP-MACAdam-corrected-kappa epoch 5: 100%|██████████| 98/98 [00:03<00:00, 26.02it/s]


  v      | min=4.39e-18 max=1.89e-03 mean=7.58e-05
  kappa_t (t=490) = 0.852632
  s2_hat | min=5.00e-05 max=1.84e-04 mean=5.00e-05 frac_at_floor=0.000
  b      | min=5.40e+00 max=7.47e+00 mean=5.40e+00


  Epoch 5 | Acc: 0.5831
Saved. Keys in file: ['dp-macadam-corrected-kappa']

All experiments complete.
Results saved to: cifar10_macadam_variants_sigma0_6.json


# Multi-seed

In [ ]:
# ── Run: single sigma, single seed, all 3 algorithm variants ─────────────────
# Set sigmas / seeds to lists with more values to reproduce a full sweep.
sigmas = [0.5, 0.6, 0.8, 1.1, 1.5]   # e.g. [0.6]
seeds  = [42, 83, 94, 100, 110]             # single seed for a quick test run

NUM_EPOCHS = 5

algorithms = {
    # "dp-macadam-paper":          run_dpadam_adaclip_paper,           # paper's algorithm exactly
    "dp-macadam-corrected-kappa": run_dpadam_adaclip_corrected_kappa, # paper's v-fix + exact kappa_t
}

for sigma in sigmas:
    print(f"\nNoise scale: {sigma}")
    NOISE_MULT = sigma
    RESULTS_PATH = f"cifar10_macadam_variants_sigma{str(sigma).replace('.', '_')}.json"

    for algo_name, algo_fn in algorithms.items():
        print(f"\n{'='*60}")
        print(f"Algorithm: {algo_name}")
        print(f"{'='*60}")
        all_accs = []
        for seed in seeds:
            print(f"\n--- Seed {seed} ---")
            accs = algo_fn(seed)
            all_accs.append(accs)
        save_results({algo_name: all_accs}, RESULTS_PATH)
        free_memory()

    print("\n" + "="*60)
    print("All experiments complete.")
    print(f"Results saved to: {RESULTS_PATH}")
    print("="*60)



Noise scale: 0.5

Algorithm: dp-macadam-corrected-kappa

--- Seed 42 ---


  DP-MACAdam-corrected-kappa epoch 1: 100%|██████████| 98/98 [00:03<00:00, 25.92it/s]


  v      | min=5.00e-16 max=4.31e-03 mean=5.29e-05
  kappa_t (t=98) = 0.852565
  s2_hat | min=5.00e-05 max=7.17e-04 mean=5.00e-05 frac_at_floor=0.000
  b      | min=5.40e+00 max=1.05e+01 mean=5.40e+00


  Epoch 1 | Acc: 0.4663


  DP-MACAdam-corrected-kappa epoch 2: 100%|██████████| 98/98 [00:03<00:00, 25.78it/s]

...


  v      | min=7.76e-12 max=2.05e+02 mean=9.49e+00
  kappa_t (t=392) = 0.852632
  s2_hat | min=5.00e-05 max=1.00e+00 mean=3.48e-03 frac_at_floor=0.000
  b      | min=6.66e+00 max=7.92e+01 mean=6.94e+00


  Epoch 4 | Acc: 0.2976


  DP-MACAdam-corrected-kappa epoch 5: 100%|██████████| 98/98 [00:03<00:00, 26.30it/s]


  v      | min=5.31e-11 max=2.03e+02 mean=9.48e+00
  kappa_t (t=490) = 0.852632
  s2_hat | min=5.00e-05 max=1.00e+00 mean=3.35e-03 frac_at_floor=0.000
  b      | min=6.62e+00 max=7.87e+01 mean=6.89e+00


  Epoch 5 | Acc: 0.2839
Saved. Keys in file: ['dp-macadam-corrected-kappa']

All experiments complete.
Results saved to: cifar10_macadam_variants_sigma1_5.json


In [ ]:
# ── Summary: mean ± std across seeds, per sigma ──────────────────────────────
for sigma in sigmas:
    results_file = f"cifar10_macadam_variants_sigma{str(sigma).replace('.', '_')}.json"
    if not os.path.exists(results_file):
        print(f"\n[missing] {results_file}")
        continue

    with open(results_file, "r") as f:
        results = json.load(f)

    print(f"\n{results_file}")
    print(f"{'Algorithm':<30} {'Final Acc (mean ± std)':>25}")
    print("-" * 57)
    for algo, all_seeds in results.items():
        final_accs = [seed_accs[-1] for seed_accs in all_seeds]  # last epoch's acc per seed
        mean = np.mean(final_accs)
        std  = np.std(final_accs)
        n    = len(final_accs)
        print(f"{algo:<30} {mean:.4f} ± {std:.4f}  (n={n})")



cifar10_macadam_variants_sigma0_5.json
Algorithm                         Final Acc (mean ± std)
---------------------------------------------------------
dp-macadam-corrected-kappa     0.5943 ± 0.0065  (n=5)

cifar10_macadam_variants_sigma0_6.json
Algorithm                         Final Acc (mean ± std)
---------------------------------------------------------
dp-macadam-corrected-kappa     0.5812 ± 0.0058  (n=5)

cifar10_macadam_variants_sigma0_8.json
Algorithm                         Final Acc (mean ± std)
---------------------------------------------------------
dp-macadam-corrected-kappa     0.5557 ± 0.0058  (n=5)

cifar10_macadam_variants_sigma1_1.json
Algorithm                         Final Acc (mean ± std)
---------------------------------------------------------
dp-macadam-corrected-kappa     0.5238 ± 0.0050  (n=5)

cifar10_macadam_variants_sigma1_5.json
Algorithm                         Final Acc (mean ± std)
---------------------------------------------------------
dp-macada